# Лабораторная работа 4: Хеширование

Решения всех заданий


## Задание 1: HashTable с квадратичным пробированием


In [ ]:
class HashTable:
    def __init__(self, size=11):
        self.size = size
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0  # количество элементов
    
    def _is_prime(self, n):
        """Проверка, является ли число простым"""
        if n < 2:
            return False
        for i in range(2, int(n ** 0.5) + 1):
            if n % i == 0:
                return False
        return True
    
    def _next_prime(self, n):
        """Находит ближайшее простое число >= n"""
        while not self._is_prime(n):
            n += 1
        return n
    
    def _resize(self, new_size):
        """Изменение размера таблицы"""
        old_slots = self.slots
        old_data = self.data
        
        self.size = new_size
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0
        
        # Перехеширование всех элементов
        for i in range(len(old_slots)):
            if old_slots[i] is not None:
                self.put(old_slots[i], old_data[i])
    
    def hashfunction(self, key):
        return key % self.size
    
    def rehash(self, oldhash, step):
        """Квадратичное пробирование"""
        return (oldhash + step * step) % self.size
    
    def put(self, key, data):
        # Проверка загрузочного фактора
        if self.count > 0 and self.count / self.size > 0.7:
            new_size = self._next_prime(self.size * 2)
            self._resize(new_size)
        
        hashvalue = self.hashfunction(key)
        
        if self.slots[hashvalue] is None:
            self.slots[hashvalue] = key
            self.data[hashvalue] = data
            self.count += 1
        elif self.slots[hashvalue] == key:
            self.data[hashvalue] = data  # обновление
        else:
            # Квадратичное пробирование
            step = 1
            nextslot = self.rehash(hashvalue, step)
            
            while self.slots[nextslot] is not None and self.slots[nextslot] != key:
                step += 1
                nextslot = self.rehash(hashvalue, step)
                
                # Проверка на зацикливание (если таблица заполнена)
                if step > self.size:
                    raise Exception("Таблица переполнена")
            
            if self.slots[nextslot] is None:
                self.slots[nextslot] = key
                self.data[nextslot] = data
                self.count += 1
            else:
                self.data[nextslot] = data  # обновление
    
    def get(self, key):
        startslot = self.hashfunction(key)
        data = None
        stop = False
        found = False
        position = startslot
        step = 0
        
        while self.slots[position] is not None and not found and not stop:
            if self.slots[position] == key:
                found = True
                data = self.data[position]
            else:
                step += 1
                position = self.rehash(startslot, step)
                if position == startslot:
                    stop = True
        
        return data
    
    def __getitem__(self, key):
        return self.get(key)
    
    def __setitem__(self, key, data):
        self.put(key, data)
    
    def __len__(self):
        return self.count
    
    def __contains__(self, key):
        return self.get(key) is not None
    
    def __delitem__(self, key):
        startslot = self.hashfunction(key)
        found = False
        position = startslot
        step = 0
        
        while self.slots[position] is not None and not found:
            if self.slots[position] == key:
                found = True
                self.slots[position] = None
                self.data[position] = None
                self.count -= 1
            else:
                step += 1
                position = self.rehash(startslot, step)
                if position == startslot:
                    break
        
        if not found:
            raise KeyError(f"Ключ {key} не найден")
        
        # Проверка загрузочного фактора для уменьшения размера
        if self.size > 11 and self.count > 0 and self.count / self.size < 0.2:
            new_size = self._next_prime(self.size // 2)
            if new_size < 11:
                new_size = 11
            self._resize(new_size)


In [ ]:
# Тестирование задания 1
h = HashTable()

# Тест добавления
h[54] = 'cat'
h[26] = 'dog'
h[93] = 'lion'
h[17] = 'tiger'
h[77] = 'bird'
h[31] = 'cow'

print(f"Размер таблицы: {len(h)}")
print(f"54 in h: {54 in h}")
print(f"h[54] = {h[54]}")

# Тест удаления
del h[54]
print(f"После удаления 54, размер: {len(h)}")
print(f"54 in h: {54 in h}")

# Тест автоматического изменения размера
for i in range(20):
    h[i] = f"value_{i}"
print(f"После добавления 20 элементов, размер таблицы: {h.size}")


## Задание 2: HashTable с методом цепочек


In [ ]:
# Реализация UnorderedList для метода цепочек
class Node:
    def __init__(self, initdata):
        self.data = initdata
        self.next = None

class UnorderedList:
    def __init__(self):
        self.head = None
        self.length = 0
    
    def add(self, item):
        temp = Node(item)
        temp.next = self.head
        self.head = temp
        self.length += 1
    
    def remove(self, item):
        current = self.head
        previous = None
        found = False
        
        while not found and current is not None:
            if current.data == item:
                found = True
            else:
                previous = current
                current = current.next
        
        if found:
            if previous is None:
                self.head = current.next
            else:
                previous.next = current.next
            self.length -= 1
            return True
        return False
    
    def search(self, item):
        current = self.head
        found = False
        while current is not None and not found:
            if current.data == item:
                found = True
            else:
                current = current.next
        return found
    
    def __len__(self):
        return self.length
    
    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.data
            current = current.next


In [ ]:
class HashTableChaining:
    def __init__(self, size=11):
        self.size = size
        self.slots = [UnorderedList() for _ in range(self.size)]
        self.count = 0
    
    def _is_prime(self, n):
        if n < 2:
            return False
        for i in range(2, int(n ** 0.5) + 1):
            if n % i == 0:
                return False
        return True
    
    def _next_prime(self, n):
        while not self._is_prime(n):
            n += 1
        return n
    
    def _resize(self, new_size):
        old_slots = self.slots
        
        self.size = new_size
        self.slots = [UnorderedList() for _ in range(self.size)]
        old_count = self.count
        self.count = 0
        
        # Перехеширование всех элементов
        for slot in old_slots:
            for key, value in slot:
                self.put(key, value)
    
    def hashfunction(self, key):
        return key % self.size
    
    def put(self, key, data):
        # Проверка загрузочного фактора
        if self.count > 0 and self.count / self.size > 0.7:
            new_size = self._next_prime(self.size * 2)
            self._resize(new_size)
        
        hashvalue = self.hashfunction(key)
        slot = self.slots[hashvalue]
        
        # Проверка, существует ли уже такой ключ
        found = False
        current = slot.head
        while current is not None:
            if current.data[0] == key:
                current.data = (key, data)  # обновление
                found = True
                break
            current = current.next
        
        if not found:
            slot.add((key, data))
            self.count += 1
    
    def get(self, key):
        hashvalue = self.hashfunction(key)
        slot = self.slots[hashvalue]
        
        current = slot.head
        while current is not None:
            if current.data[0] == key:
                return current.data[1]
            current = current.next
        
        return None
    
    def __getitem__(self, key):
        return self.get(key)
    
    def __setitem__(self, key, data):
        self.put(key, data)
    
    def __len__(self):
        return self.count
    
    def __contains__(self, key):
        return self.get(key) is not None
    
    def __delitem__(self, key):
        hashvalue = self.hashfunction(key)
        slot = self.slots[hashvalue]
        
        # Поиск и удаление элемента по ключу
        current = slot.head
        previous = None
        found = False
        
        while current is not None and not found:
            if current.data[0] == key:
                found = True
                if previous is None:
                    slot.head = current.next
                else:
                    previous.next = current.next
                slot.length -= 1
                self.count -= 1
            else:
                previous = current
                current = current.next
        
        if not found:
            raise KeyError(f"Ключ {key} не найден")
        
        # Проверка загрузочного фактора для уменьшения размера
        if self.size > 11 and self.count > 0 and self.count / self.size < 0.2:
            new_size = self._next_prime(self.size // 2)
            if new_size < 11:
                new_size = 11
            self._resize(new_size)


In [ ]:
# Тестирование задания 2
h2 = HashTableChaining()

h2[54] = 'cat'
h2[26] = 'dog'
h2[93] = 'lion'
h2[17] = 'tiger'

print(f"Размер таблицы: {len(h2)}")
print(f"54 in h2: {54 in h2}")
print(f"h2[54] = {h2[54]}")

del h2[54]
print(f"После удаления 54, размер: {len(h2)}")
print(f"54 in h2: {54 in h2}")


## Задание 3: HashTable для работы со строками


In [ ]:
class HashTableString:
    def __init__(self, size=11):
        self.size = size
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0
    
    def _is_prime(self, n):
        if n < 2:
            return False
        for i in range(2, int(n ** 0.5) + 1):
            if n % i == 0:
                return False
        return True
    
    def _next_prime(self, n):
        while not self._is_prime(n):
            n += 1
        return n
    
    def _resize(self, new_size):
        old_slots = self.slots
        old_data = self.data
        
        self.size = new_size
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0
        
        for i in range(len(old_slots)):
            if old_slots[i] is not None:
                self.put(old_slots[i], old_data[i])
    
    def hashfunction(self, key):
        """Хеш-функция для строк"""
        if isinstance(key, str):
            hash_value = 0
            for char in key:
                hash_value = (hash_value * 31 + ord(char)) % self.size
            return hash_value
        else:
            return key % self.size
    
    def rehash(self, oldhash, step):
        return (oldhash + step * step) % self.size
    
    def put(self, key, data):
        if self.count > 0 and self.count / self.size > 0.7:
            new_size = self._next_prime(self.size * 2)
            self._resize(new_size)
        
        hashvalue = self.hashfunction(key)
        
        if self.slots[hashvalue] is None:
            self.slots[hashvalue] = key
            self.data[hashvalue] = data
            self.count += 1
        elif self.slots[hashvalue] == key:
            self.data[hashvalue] = data
        else:
            step = 1
            nextslot = self.rehash(hashvalue, step)
            
            while self.slots[nextslot] is not None and self.slots[nextslot] != key:
                step += 1
                nextslot = self.rehash(hashvalue, step)
                if step > self.size:
                    raise Exception("Таблица переполнена")
            
            if self.slots[nextslot] is None:
                self.slots[nextslot] = key
                self.data[nextslot] = data
                self.count += 1
            else:
                self.data[nextslot] = data
    
    def get(self, key):
        startslot = self.hashfunction(key)
        data = None
        stop = False
        found = False
        position = startslot
        step = 0
        
        while self.slots[position] is not None and not found and not stop:
            if self.slots[position] == key:
                found = True
                data = self.data[position]
            else:
                step += 1
                position = self.rehash(startslot, step)
                if position == startslot:
                    stop = True
        
        return data
    
    def __getitem__(self, key):
        return self.get(key)
    
    def __setitem__(self, key, data):
        self.put(key, data)
    
    def __len__(self):
        return self.count
    
    def __contains__(self, key):
        return self.get(key) is not None
    
    def __delitem__(self, key):
        startslot = self.hashfunction(key)
        found = False
        position = startslot
        step = 0
        
        while self.slots[position] is not None and not found:
            if self.slots[position] == key:
                found = True
                self.slots[position] = None
                self.data[position] = None
                self.count -= 1
            else:
                step += 1
                position = self.rehash(startslot, step)
                if position == startslot:
                    break
        
        if not found:
            raise KeyError(f"Ключ {key} не найден")
        
        if self.size > 11 and self.count > 0 and self.count / self.size < 0.2:
            new_size = self._next_prime(self.size // 2)
            if new_size < 11:
                new_size = 11
            self._resize(new_size)


In [ ]:
# Тестирование задания 3
h3 = HashTableString()

h3['яблоко'] = 'apple'
h3['банан'] = 'banana'
h3['апельсин'] = 'orange'

print(f"Размер: {len(h3)}")
print(f"'яблоко' in h3: {'яблоко' in h3}")
print(f"h3['яблоко'] = {h3['яблоко']}")


## Задание 4: Подсчет порядковых номеров вхождений слов


In [ ]:
def count_word_occurrences(text):
    """
    Для каждого слова текста выводит порядковый номер его вхождения.
    """
    words = text.split()
    word_count = HashTableString()
    result = []
    
    for word in words:
        if word in word_count:
            # Увеличиваем счетчик
            current_count = word_count[word]
            word_count[word] = current_count + 1
            result.append(str(current_count + 1))
        else:
            # Первое вхождение
            word_count[word] = 1
            result.append('1')
    
    return ' '.join(result)

# Тест
text = "Раз раз раз как меня слышно"
result = count_word_occurrences(text)
print(f"Ввод: {text}")
print(f"Вывод: {result}")


## Задание 5: Регистрация и авторизация с хешированием паролей


In [ ]:
import hashlib
import secrets
import json
import os

class AuthSystem:
    def __init__(self, db_file='users.json'):
        self.db_file = db_file
        self.users = self._load_users()
    
    def _load_users(self):
        """Загрузка пользователей из файла"""
        if os.path.exists(self.db_file):
            with open(self.db_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        return {}
    
    def _save_users(self):
        """Сохранение пользователей в файл"""
        with open(self.db_file, 'w', encoding='utf-8') as f:
            json.dump(self.users, f, ensure_ascii=False, indent=2)
    
    def register(self, login, password):
        """Регистрация нового пользователя"""
        if login in self.users:
            return False, "Пользователь с таким логином уже существует"
        
        # Генерация соли (32 байта для SHA256)
        salt = secrets.token_bytes(32)
        
        # Хеширование пароля с солью
        password_hash = hashlib.sha256(password.encode('utf-8') + salt).hexdigest()
        
        # Сохранение (соль в hex формате для JSON)
        self.users[login] = {
            'salt': salt.hex(),
            'hash': password_hash
        }
        
        self._save_users()
        return True, "Пользователь успешно зарегистрирован"
    
    def login(self, login, password):
        """Авторизация пользователя"""
        if login not in self.users:
            return False, "Пользователь не найден"
        
        user_data = self.users[login]
        salt = bytes.fromhex(user_data['salt'])
        stored_hash = user_data['hash']
        
        # Хеширование введенного пароля с той же солью
        password_hash = hashlib.sha256(password.encode('utf-8') + salt).hexdigest()
        
        if password_hash == stored_hash:
            return True, "Авторизация успешна"
        else:
            return False, "Неверный пароль"

# Тестирование
auth = AuthSystem('test_users.json')

# Регистрация
success, message = auth.register('user1', 'password123')
print(f"Регистрация: {message}")

# Авторизация с правильным паролем
success, message = auth.login('user1', 'password123')
print(f"Авторизация (правильный пароль): {message}")

# Авторизация с неправильным паролем
success, message = auth.login('user1', 'wrongpassword')
print(f"Авторизация (неправильный пароль): {message}")

# Очистка тестового файла
if os.path.exists('test_users.json'):
    os.remove('test_users.json')


## Задание 6: Поиск дубликатов файлов


In [ ]:
import hashlib
import os
from collections import defaultdict

def find_duplicates(directory_path):
    """
    Находит все дубликаты файлов в указанной директории.
    """
    if not os.path.exists(directory_path):
        print(f"Директория {directory_path} не существует")
        return
    
    if not os.path.isdir(directory_path):
        print(f"{directory_path} не является директорией")
        return
    
    # Словарь: хеш -> список путей к файлам
    file_hashes = defaultdict(list)
    
    # Обход всех файлов в директории
    for root, dirs, files in os.walk(directory_path):
        for filename in files:
            filepath = os.path.join(root, filename)
            
            try:
                # Вычисление SHA256 хеша файла
                sha256_hash = hashlib.sha256()
                with open(filepath, "rb") as f:
                    # Чтение файла блоками для экономии памяти
                    for byte_block in iter(lambda: f.read(4096), b""):
                        sha256_hash.update(byte_block)
                
                file_hash = sha256_hash.hexdigest()
                file_hashes[file_hash].append(filepath)
                
            except (IOError, OSError) as e:
                print(f"Ошибка при чтении файла {filepath}: {e}")
    
    # Поиск дубликатов (файлы с одинаковым хешем)
    duplicates_found = False
    for file_hash, filepaths in file_hashes.items():
        if len(filepaths) > 1:
            duplicates_found = True
            print(f"\nДубликаты (хеш: {file_hash[:16]}...):")
            for filepath in filepaths:
                print(f"  - {filepath}")
    
    if not duplicates_found:
        print("Дубликаты не найдены")

# Пример использования
# find_duplicates('C:/путь/к/директории')
print("Функция find_duplicates готова к использованию")
print("Вызовите: find_duplicates('путь/к/директории')")
